# RSNC waveform acquisition

Downloads event-station waveform windows from the FDSN service and writes SAC files under `data/raw/waveforms/`. Run after the catalogue is available.


In [1]:
import time
import pandas as pd
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException
from obspy.taup import TauPyModel
from obspy.geodetics import gps2dist_azimuth, locations2degrees
from obspy.core.util import AttribDict

## 1. Configuration and paths

Choose the stations, event filters, distance limit, and window around the theoretical P arrival. The current station selection is `ZAR`; update it for another acquisition run. Project paths are detected from the working directory.

In [2]:
# ==========================================
# 1. CONFIGURATION AND INPUT VARIABLES
# ==========================================
STATIONS_TO_PROCESS = ["ZAR"]  # Add more, e.g. ["PTB", "CHI", "PAL"]
MIN_MAGNITUDE = 1.5            # Minimum event magnitude
START_DATE = "2018-07-22"      # Start date filter
END_DATE = "2025-12-31"        # End date filter
MAX_DISTANCE_DEG = 3.0         # Max epicentral distance (degrees)

# Cut window (seconds relative to the theoretical P arrival)
TIME_BEFORE = 30
TIME_AFTER = 120

# FDSN request settings
NETWORK = "CM"
LOCATION = "00"
CHANNELS = "HH?"

# Retry / rate-limit settings (lower timeout => failures resolve much faster)
REQUEST_TIMEOUT = 30     # seconds per FDSN request before giving up
MAX_RETRIES = 3          # network retries per request
RETRY_BACKOFF = 3        # seconds; wait between retries = RETRY_BACKOFF * attempt
SLEEP_BETWEEN_OK = 0.5   # polite pause (s) after a successful download

model = TauPyModel(model="iasp91")
client = Client("http://sismo.sgc.gov.co:8080", timeout=REQUEST_TIMEOUT)

# ==========================================
# PROJECT PATHS (aligned with the repo structure)
# ==========================================
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk up from the current directory until the repo root is found
    (the folder that contains data/raw and src)."""
    for p in (start, *start.parents):
        if (p / "data" / "raw").is_dir() and (p / "src").is_dir():
            return p
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
CATALOG_DIR = REPO_ROOT / "data" / "raw" / "catalog"       # inputs: station and event CSVs
WAVEFORMS_DIR = REPO_ROOT / "data" / "raw" / "waveforms"   # output: SAC per station
WAVEFORMS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:  {REPO_ROOT}")
print(f"Catalog:    {CATALOG_DIR}")
print(f"SAC output: {WAVEFORMS_DIR}")

## 2. Load and filter the catalogs

Read station coordinates and RSNC events from `data/raw/catalog/`, then retain events within the configured magnitude and date ranges.

In [3]:
# ==========================================
# 2. LOAD AND FILTER WITH PANDAS
# ==========================================
print("Loading and filtering databases...")

# Load stations
df_stations = pd.read_csv(CATALOG_DIR / 'Stations_Required.csv')
df_stations.columns = [c.strip() for c in df_stations.columns]

# Load events
df_events = pd.read_csv(CATALOG_DIR / 'Events_2018_2025.csv', encoding='utf-8-sig')
df_events.columns = [c.strip() for c in df_events.columns]

# Convert and filter by date and magnitude
df_events['FECHA_DT'] = pd.to_datetime(df_events['FECHA - HORA UTC'])
mask = (
    (df_events['MAGNITUD'] >= MIN_MAGNITUDE) & 
    (df_events['FECHA_DT'] >= pd.to_datetime(START_DATE)) & 
    (df_events['FECHA_DT'] <= pd.to_datetime(END_DATE))
)
df_filtered = df_events.loc[mask].copy()

print(f"Events passing Mag and Date filter: {len(df_filtered)}")

Cargando y filtrando bases de datos...
Eventos que cumplen filtro de Mag y Fecha: 117285


## 3. Download and save waveforms

For each eligible station–event pair, estimate the theoretical P arrival, request the three HH components, and save the event window as SAC with event geometry in its header. Existing windows are skipped, and failed requests are retried.

In [ ]:
# ==========================================
# 3. ROBUST, RESUMABLE DOWNLOAD
# ==========================================
n_ok = n_empty = n_fail = n_skip = n_far = n_bad = 0

for sta_code in STATIONS_TO_PROCESS:
    sta_row = df_stations[df_stations["Code"] == sta_code]
    if sta_row.empty:
        print(f"[SKIP] Station {sta_code} not found in the station catalog.")
        continue
    stalat = float(sta_row.iloc[0]["Latitude"])
    stalon = float(sta_row.iloc[0]["Longitude"])
    staelev = float(sta_row.iloc[0]["Elevation"])

    sta_dir = WAVEFORMS_DIR / sta_code
    sta_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n--- Processing station: {sta_code} ---")

    for _, row in df_filtered.iterrows():
        orig_time = UTCDateTime(row["FECHA - HORA UTC"])
        evlat, evlon, evdepth = row["LATITUD (°)"], row["LONGITUD (°)"], row["PROF. (Km)"]

        # 1) Epicentral distance (obspy); skip far pairs before any heavy work
        dist_deg = locations2degrees(stalat, stalon, evlat, evlon)
        if dist_deg > MAX_DISTANCE_DEG:
            n_far += 1
            continue

        # 2) Validate depth, then theoretical P arrival (computed once, no network)
        if pd.isna(evdepth) or evdepth < 0:
            n_bad += 1
            continue
        try:
            arrivals = model.get_travel_times(
                source_depth_in_km=evdepth + (staelev / 1000),
                distance_in_degree=dist_deg,
                phase_list=["P"],
            )
        except Exception as e:
            print(f" [SKIP] {orig_time}: travel-time error ({e}).")
            n_bad += 1
            continue
        if not arrivals:
            continue

        p_arrival = orig_time + arrivals[0].time
        t1, t2 = p_arrival - TIME_BEFORE, p_arrival + TIME_AFTER
        t_stamp = t1.strftime("%Y.%j.%H.%M.%S")

        # 3) Resume support: if this window is already downloaded, skip it
        if list(sta_dir.glob(f"{sta_code}.*.{t_stamp}.SAC")):
            n_skip += 1
            continue

        # 4) Back-azimuth for the SAC header
        _, _, baz = gps2dist_azimuth(stalat, stalon, evlat, evlon)

        # 5) Download (only the network call is retried)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                st = client.get_waveforms(NETWORK, sta_code, LOCATION, CHANNELS, t1, t2)
                for tr in st:
                    tr.stats.sac = AttribDict({
                        "dist": dist_deg, "baz": baz,
                        "evla": evlat, "evlo": evlon, "evdp": evdepth,
                    })
                    fname = str(sta_dir / f"{sta_code}.{tr.stats.channel}.{t_stamp}.SAC")
                    tr.write(fname, format="SAC")
                print(f" [OK] {orig_time} (attempt {attempt})")
                n_ok += 1
                time.sleep(SLEEP_BETWEEN_OK)
                break
            except FDSNNoDataException:
                print(f" [EMPTY] {orig_time}: no data.")
                n_empty += 1
                break
            except Exception as e:
                if attempt == MAX_RETRIES:
                    print(f" [FAIL] {orig_time} after {MAX_RETRIES} attempts: {e}")
                    n_fail += 1
                else:
                    time.sleep(RETRY_BACKOFF * attempt)

print(f"\nDone. OK={n_ok}  empty={n_empty}  failed={n_fail}  "
      f"skipped(existing)={n_skip}  too_far={n_far}  bad_meta={n_bad}")


--- Iniciando Procesamiento: ZAR ---
 [OK] 2018-07-22T00:12:02.000000Z (Intento 1)
 [FALLO] Saltando 2018-07-22T00:17:17.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T01:32:16.000000Z después de 3 intentos.
 [OK] 2018-07-22T01:50:25.000000Z (Intento 1)
 [OK] 2018-07-22T01:50:44.000000Z (Intento 1)
 [FALLO] Saltando 2018-07-22T01:58:18.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T02:35:40.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T03:12:39.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T03:46:16.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T03:49:18.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T04:15:44.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T04:44:15.000000Z después de 3 intentos.
 [OK] 2018-07-22T04:54:54.000000Z (Intento 1)
 [FALLO] Saltando 2018-07-22T05:20:19.000000Z después de 3 intentos.
 [FALLO] Saltando 2018-07-22T05:29:32.000000Z después de 3 intentos.
 [FALLO] Saltando 2

## Optional connection check

This standalone request checks the FDSN connection. It is not part of the acquisition loop; a no-data response can reflect the selected station and time.

In [93]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("http://sismo.sgc.gov.co:8080")

t1 = UTCDateTime("2018-03-10T12:00:00")
t2 = t1 + 300

st = client.get_waveforms(
    network="CM",
    station="EC08",
    location="*",
    channel="BH?",
    starttime=t1,
    endtime=t2
)

print(st)


FDSNNoDataException: No data available for request.
HTTP Status code: 204
Detailed response of server:

